# 🏙️ UrbanHub - Jumeau Numérique Urbain
### Plateforme Big Data & Tableau de Bord Smart City (Restitution Gold)

> **Architecture :** Data Lake Medallion (Couche `Gold` Cloudflare R2 / MinIO)  
> **Flux Ingestion :** 🌦️ Météo Batch (NOAA 5 ans) | 🚲 Mobilité Streaming (CityBikes) | 🌫️ Pollution IoT (OpenAQ v3)

In [ ]:
import os
import io
import json
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from minio import Minio

# 1. Chargement robuste du .env
def load_env():
    candidates = ["../.env", ".env", "/app/.env"]
    for p in candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        os.environ[k.strip()] = v.strip().strip("'\"")
            return p
    return None

load_env()

# 2. Chargement du référentiel des villes
config_path = "../config/cities.yml" if os.path.exists("../config/cities.yml") else "config/cities.yml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

cities_meta = {c["id"]: c for c in config.get("cities", [])}
cities_dict = {c["name"]: c["id"] for c in config.get("cities", [])}

# 3. Connexion S3 (MinIO local ou Cloudflare R2)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"
BUCKET_GOLD = os.getenv("MINIO_BUCKET_GOLD", "urbanhub-gold")

client = Minio(
    MINIO_ENDPOINT,
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    secure=SECURE
)

def read_parquet_table(keys):
    if isinstance(keys, str):
        keys = [keys]
    for k in keys:
        try:
            resp = client.get_object(BUCKET_GOLD, k)
            df = pd.read_parquet(io.BytesIO(resp.read()))
            resp.close()
            resp.release_conn()
            return df
        except Exception:
            continue
    return None

def read_json_meta(key):
    try:
        resp = client.get_object(BUCKET_GOLD, key)
        data = json.loads(resp.read().decode('utf-8'))
        resp.close()
        resp.release_conn()
        return data
    except Exception:
        return None

print(f"✅ Data Lake connecté : {'https' if SECURE else 'http'}://{MINIO_ENDPOINT} (Bucket Gold : {BUCKET_GOLD})")

## 🎛️ Sélecteurs de Supervision Urbaine

In [ ]:
city_selector = widgets.Dropdown(
    options=list(cities_dict.keys()),
    value=list(cities_dict.keys())[0] if cities_dict else None,
    description="🏙️ Ville :",
    layout=widgets.Layout(width="320px")
)

refresh_btn = widgets.Button(
    description="Actualiser les KPI",
    button_style="primary",
    icon="sync",
    layout=widgets.Layout(width="200px")
)

output_dashboard = widgets.Output()

controls_box = widgets.HBox(
    [city_selector, refresh_btn],
    layout=widgets.Layout(margin="10px 0 20px 0", align_items="center")
)
display(controls_box)
display(output_dashboard)

## 📊 Restitution & Visualisations Multidimensionnelles

In [ ]:
def render_dashboard(b=None):
    selected_name = city_selector.value
    selected_id = cities_dict.get(selected_name, "paris")
    city_info = cities_meta.get(selected_id, {"latitude": 48.8566, "longitude": 2.3522})
    
    with output_dashboard:
        clear_output(wait=True)
        
        # En-tête dynamique
        header_html = f"""
        <div style='background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 18px 25px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <div style='display: flex; justify-content: space-between; align-items: center;'>
                <div>
                    <h2 style='margin: 0; font-size: 24px;'>Supervision Métropolitaine : {selected_name}</h2>
                    <p style='margin: 5px 0 0 0; opacity: 0.85; font-size: 14px;'>Identifiant : <code>{selected_id}</code> | Coordonnées : {city_info.get('latitude', 0):.4f}°N, {city_info.get('longitude', 0):.4f}°E</p>
                </div>
                <div style='text-align: right;'>
                    <span style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px; font-weight: bold; font-size: 13px;'>🟢 Data Lake Connecté</span>
                </div>
            </div>
        </div>
        """
        display(HTML(header_html))
        
        # Chargement des tables
        df_w = read_parquet_table(["kpis/weather_daily.parquet"])
        df_m = read_parquet_table(["citybikes/kpis/mobility_station_latest.parquet", "kpis/mobility_station_latest.parquet"])
        df_m_hourly = read_parquet_table(["citybikes/kpis/mobility_hourly.parquet", "kpis/mobility_hourly.parquet"])
        df_p = read_parquet_table(["kpis/pollution_hourly.parquet"])
        p_summary = read_json_meta("kpis/pollution_summary.json")
        df_c = read_parquet_table(["kpis/city_hourly.parquet"])
        
        # Création des onglets
        tab = widgets.Tab()
        tab_titles = ["🌦️ Météo (NOAA)", "🚲 Mobilité Douce (CityBikes)", "🌫️ Qualité de l'Air (OpenAQ)", "🔄 Jumeau & Croisement"]
        
        # ==========================================================
        # ONGLET 1 : MÉTÉO
        # ==========================================================
        out_w = widgets.Output()
        with out_w:
            if df_w is not None and not df_w.empty:
                sub_w = df_w[df_w["city_id"] == selected_id].copy()
                if not sub_w.empty:
                    t_mean = sub_w["temperature_mean"].mean()
                    t_min = sub_w["temperature_min"].min()
                    t_max = sub_w["temperature_max"].max()
                    w_max = sub_w["wind_speed_max"].max()
                    nb_days = len(sub_w)
                    
                    kpi_w_html = f"""
                    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px;'>
                        <div style='background: #f8f9fa; border-left: 5px solid #3498db; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>TEMPÉRATURE MOYENNE</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2c3e50; margin-top: 5px;'>{t_mean:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Sur {nb_days} jours observés</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #2ecc71; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>EXTRÊME MINIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #27ae60; margin-top: 5px;'>{t_min:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Record froid observé</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #e74c3c; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>EXTRÊME MAXIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #c0392b; margin-top: 5px;'>{t_max:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Record chaleur observé</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #f39c12; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>VENT MAXIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #d35400; margin-top: 5px;'>{w_max:.1f} m/s</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Rafale maximale enregistrée</div>
                        </div>
                    </div>
                    """
                    display(HTML(kpi_w_html))
                    
                    # Graphique d'évolution temporelle
                    fig, ax = plt.subplots(figsize=(11, 4), dpi=100)
                    sub_w = sub_w.sort_values("day_utc").tail(180) # Derniers 6 mois pour lisibilité
                    ax.plot(pd.to_datetime(sub_w["day_utc"]), sub_w["temperature_mean"], label="Température Moyenne (°C)", color="#2980b9", lw=2)
                    ax.fill_between(pd.to_datetime(sub_w["day_utc"]), sub_w["temperature_min"], sub_w["temperature_max"], color="#3498db", alpha=0.2, label="Amplitude Min-Max")
                    ax.set_title(f"Chronologie des Températures Journalières - {selected_name} (Derniers 180 jours)", fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Date")
                    ax.set_ylabel("Température (°C)")
                    ax.grid(True, linestyle="--", alpha=0.5)
                    ax.legend(loc="upper left")
                    plt.tight_layout()
                    plt.show()
                else:
                    display(HTML(f"<p style='color:#e67e22;'>⚠️ Aucune donnée météo trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>❌ Table Gold météo introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 2 : MOBILITÉ CITYBIKES
        # ==========================================================
        out_m = widgets.Output()
        with out_m:
            if df_m is not None and not df_m.empty:
                sub_m = df_m[df_m["city_id"] == selected_id].copy()
                if not sub_m.empty:
                    tot_stations = len(sub_m)
                    tot_bikes = int(sub_m["bikes_available"].sum())
                    tot_slots = int(sub_m["free_slots"].sum())
                    nb_empty = int((sub_m["bikes_available"] == 0).sum())
                    nb_full = int((sub_m["free_slots"] == 0).sum())
                    pct_dispo = (tot_bikes / (tot_bikes + tot_slots) * 100) if (tot_bikes + tot_slots) > 0 else 0
                    
                    kpi_m_html = f"""
                    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px;'>
                        <div style='background: #f8f9fa; border-left: 5px solid #27ae60; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>TOTAL STATIONS</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2c3e50; margin-top: 5px;'>{tot_stations}</div>
                            <div style='color: #27ae60; font-size: 12px; margin-top: 3px;'>Réseau actif en direct</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #2980b9; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>VÉLOS DISPONIBLES</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2980b9; margin-top: 5px;'>{tot_bikes}</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>{pct_dispo:.1f}% de taux de remplissage</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #8e44ad; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>BORNES LIBRES</div>
                            <div style='font-size: 26px; font-weight: bold; color: #8e44ad; margin-top: 5px;'>{tot_slots}</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Capacité d'accueil retour</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #e74c3c; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>STATIONS EN RUPTURE</div>
                            <div style='font-size: 26px; font-weight: bold; color: #c0392b; margin-top: 5px;'>{nb_empty}</div>
                            <div style='color: #e67e22; font-size: 12px; margin-top: 3px;'>{nb_full} stations saturées</div>
                        </div>
                    </div>
                    """
                    display(HTML(kpi_m_html))
                    
                    # Carte interactive Folium (OpenStreetMap 100% libre et sans clé API)
                    if HAS_FOLIUM:
                        display(HTML("<h4>🗺️ Cartographie Temps Réel des Stations & Niveaux de Disponibilité</h4>"))
                        m = folium.Map(
                            location=[city_info.get("latitude", 48.8566), city_info.get("longitude", 2.3522)],
                            zoom_start=13,
                            tiles="OpenStreetMap"
                        )
                        
                        # Ajout des marqueurs de stations (échantillonnage de 400 stations pour fluidité)
                        sub_sample = sub_m.head(400) if len(sub_m) > 400 else sub_m
                        for _, row in sub_sample.iterrows():
                            lat = row.get("latitude")
                            lon = row.get("longitude")
                            if pd.notna(lat) and pd.notna(lon):
                                bikes = int(row.get("bikes_available", 0))
                                slots = int(row.get("free_slots", 0))
                                name = row.get("station_name", "Station")
                                
                                if bikes == 0:
                                    color = "#e74c3c" # Rouge (rupture)
                                elif bikes < 4:
                                    color = "#f39c12" # Orange (tension)
                                else:
                                    color = "#27ae60" # Vert (dispo)
                                
                                folium.CircleMarker(
                                    location=[lat, lon],
                                    radius=5,
                                    color=color,
                                    fill=True,
                                    fill_color=color,
                                    fill_opacity=0.8,
                                    popup=f"<b>{name}</b><br>🚲 Vélos dispo : {bikes}<br>🅿️ Places libres : {slots}"
                                ).add_to(m)
                        
                        display(HTML(m._repr_html_()))
                    else:
                        display(HTML("<p style='color:#7f8c8d;'><i>Module folium non disponible pour la carte. Aperçu tabulaire :</i></p>"))
                        display(sub_m[['station_name', 'bikes_available', 'free_slots']].head(15))
                else:
                    display(HTML(f"<p style='color:#e67e22;'>⚠️ Aucune station trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>❌ Table Gold mobilité introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 3 : POLLUTION OPENAQ
        # ==========================================================
        out_p = widgets.Output()
        with out_p:
            if df_p is not None and not df_p.empty:
                sub_p = df_p[df_p["city_id"] == selected_id].copy()
                if not sub_p.empty:
                    who_thresholds = {"pm25": 15.0, "no2": 25.0, "pm10": 45.0, "o3": 100.0, "co": 4000.0}
                    stats = sub_p.groupby("pollutant")["value_mean"].mean().to_dict()
                    
                    kpi_p_cards = ""
                    for pol, val in stats.items():
                        thr = who_thresholds.get(pol, 50.0)
                        is_danger = val > thr
                        badge = "<span style='color:#e74c3c; font-weight:bold;'>⚠️ Dépassement OMS</span>" if is_danger else "<span style='color:#27ae60; font-weight:bold;'>✅ Conforme OMS</span>"
                        border_c = "#e74c3c" if is_danger else "#27ae60"
                        kpi_p_cards += f"""
                        <div style='background: #f8f9fa; border-left: 5px solid {border_c}; padding: 12px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 12px; font-weight: bold;'>POLLUANT : {pol.upper()}</div>
                            <div style='font-size: 22px; font-weight: bold; color: #2c3e50; margin-top: 4px;'>{val:.2f} µg/m³</div>
                            <div style='font-size: 11px; margin-top: 3px;'>Seuil OMS : {thr} µg/m³<br>{badge}</div>
                        </div>
                        """
                    
                    display(HTML(f"<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 12px; margin-bottom: 20px;'>{kpi_p_cards}</div>"))
                    
                    # Graphique barres comparatif
                    fig, ax = plt.subplots(figsize=(9, 4), dpi=100)
                    pols = [p for p in stats.keys() if p != "co"] # Exclusion CO car échelle différente (mg)
                    vals = [stats[p] for p in pols]
                    thrs = [who_thresholds.get(p, 50) for p in pols]
                    
                    x = np.arange(len(pols))
                    width = 0.35
                    ax.bar(x - width/2, vals, width, label="Moyenne Observée", color="#e67e22")
                    ax.bar(x + width/2, thrs, width, label="Seuil Recommandé OMS", color="#95a5a6", alpha=0.6)
                    ax.set_title(f"Conformité Qualité de l'Air vs Seuils OMS - {selected_name}", fontsize=12, fontweight="bold")
                    ax.set_xticks(x)
                    ax.set_xticklabels([p.upper() for p in pols])
                    ax.set_ylabel("Concentration (µg/m³)")
                    ax.legend()
                    ax.grid(True, linestyle="--", alpha=0.4)
                    plt.tight_layout()
                    plt.show()
                else:
                    display(HTML(f"<p style='color:#e67e22;'>⚠️ Aucune mesure de pollution trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>❌ Table Gold pollution introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 4 : ANALYSE CROISÉE & JUMEAU NUMÉRIQUE
        # ==========================================================
        out_c = widgets.Output()
        with out_c:
            if df_c is not None and not df_c.empty:
                sub_c = df_c[df_c["city_id"] == selected_id].copy()
                if not sub_c.empty:
                    nb_tot = len(sub_c)
                    nb_poll = int(sub_c["pollution_no2_ugm3"].notna().sum()) if "pollution_no2_ugm3" in sub_c else 0
                    nb_mob = int(sub_c["mobility_bikes_available"].notna().sum()) if "mobility_bikes_available" in sub_c else 0
                    nb_met = int(sub_c["weather_temperature_c"].notna().sum()) if "weather_temperature_c" in sub_c else 0
                    
                    cross_info_html = f"""
                    <div style='background: #eef2f7; border-left: 5px solid #2980b9; padding: 16px 20px; border-radius: 8px; margin-bottom: 20px;'>
                        <h4 style='margin: 0 0 8px 0; color: #2c3e50;'>ℹ️ Architecture du Jumeau Numérique & Analyse Temporelle</h4>
                        <p style='margin: 0 0 10px 0; font-size: 13px; color: #34495e;'>
                            Dans notre plateforme Smart City, chaque flux possède sa propre dynamique temporelle :
                        </p>
                        <ul style='margin: 0; padding-left: 20px; font-size: 13px; color: #2c3e50; line-height: 1.6;'>
                            <li><b>🌦️ Météo (NOAA) :</b> Historique massif sur 5 ans (2020–2024) - <b>{nb_met}</b> heures archivées.</li>
                            <li><b>🌫️ Pollution (OpenAQ) :</b> Ingestion IoT active (Novembre 2026) - <b>{nb_poll}</b> heures mesurées.</li>
                            <li><b>🚲 Mobilité (CityBikes) :</b> Micro-batch streaming temps réel (Novembre 2026) - <b>{nb_mob}</b> créneaux enregistrés.</li>
                        </ul>
                    </div>
                    """
                    display(HTML(cross_info_html))
                    
                    # Tableau filtré des observations actives récentes (sans NaN bruts)
                    has_active = pd.Series(False, index=sub_c.index)
                    if "pollution_no2_ugm3" in sub_c:
                        has_active = has_active | sub_c["pollution_no2_ugm3"].notna()
                    if "mobility_bikes_available" in sub_c:
                        has_active = has_active | sub_c["mobility_bikes_available"].notna()
                    
                    sub_active = sub_c[has_active].sort_values("hour_utc", ascending=False).copy()
                    
                    if not sub_active.empty:
                        display(HTML("<h4>🕒 Dernières Observations Multi-Sources Actives (Novembre 2026)</h4>"))
                        cols_show = ["hour_utc", "city_name", "pollution_no2_ugm3", "pollution_pm25_ugm3", "pollution_pm10_ugm3", "mobility_bikes_available", "mobility_free_slots", "statut_jointure"]
                        cols_exist = [c for c in cols_show if c in sub_active.columns]
                        table_display = sub_active[cols_exist].head(15).copy().fillna("-")
                        table_display.columns = [c.replace("pollution_", "Pollution ").replace("mobility_", "Vélos ").replace("_ugm3", " (µg/m³)").replace("hour_utc", "Date & Heure UTC") for c in table_display.columns]
                        display(table_display)
                    else:
                        display(HTML("<h4>🕒 Dernières Observations Météorologiques Enregistrées</h4>"))
                        cols_w_show = ["hour_utc", "city_name", "weather_temperature_c", "weather_wind_speed_ms", "weather_precipitation_mm"]
                        cols_w_exist = [c for c in cols_w_show if c in sub_c.columns]
                        table_w = sub_c[cols_w_exist].sort_values("hour_utc", ascending=False).head(15).copy().fillna("-")
                        display(table_w)
                else:
                    display(HTML(f"<p style='color:#e67e22;'>⚠️ Aucune heure croisée trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>❌ Table Gold de croisement introuvable.</p>"))
        
        tab.children = [out_w, out_m, out_p, out_c]
        for i, t in enumerate(tab_titles):
            tab.set_title(i, t)
        display(tab)

refresh_btn.on_click(render_dashboard)
city_selector.observe(render_dashboard, names='value')
render_dashboard()